# Human-in-the-Loop: Malango ya Kabla ya Hatua, Kiwango cha Hatari, na Kumbukumbu ya Ukaguzi

README ya somo hili inatambulisha Human-in-the-Loop kwa kipande kifupi kinachoomba mtumiaji `APPROVE` au `REJECT` baada ya wakala tayari kutoa jibu. Mfumo huo ni mwanzo mzuri, lakini utekelezaji wa HITL katika uzalishaji kawaida unahitaji vipengele vitatu vya ziada:

1. **mlango wa kabla ya hatua** unaofanya kazi **kabla** wakala hajafanya hatua yenye hatari, ili gharama, kutoweza kubadilika, na ucheleweshaji vishikwe chini ya udhibiti.
2. **Kiwango cha hatari**, kwa matendo yenye hatari ndogo kufanywa moja kwa moja, matendo yenye hatari ya kati yathibitishwe kwa kundi, na matendo yenye hatari kubwa kuzuia mtu mmoja.
3. **Rekodi ya ukaguzi pamoja na mzunguko wa marekebisho**, kwa kila uamuzi wa mlango kurekodiwa kama JSONL, na kukataliwa kuwasilisha tena wakala kwa sababu iliyopangwa badala ya kuchapisha tu `Revising...`.

Kitabu hiki cha mazoezi kinajenga kila mojawapo juu ya vitu sawa kama `06-system-message-framework.ipynb`. Kinaendeshwa kutoka mwanzo hadi mwisho katika `DEMO_MODE = True` (hazihitaji pembejeo za mtumiaji) au kwa miito halisi ya `input()` wakati `DEMO_MODE = False`. Kumbuka: katika DEMO_MODE jaribio la tatu ni la maandishi hivyo michakato ya mzunguko inaonekana hadi mwisho. Upangaji tena unaoendeshwa kwa marekebisho halisi unahitaji `DEMO_MODE = False` na operator.

**Haijajumuishwa (inashughulikiwa katika masomo mengine):** uthibitishaji na udhibiti wa ufikiaji (somosomo 06 README hatari 2), middleware ya simu za zana (somo 14 MAF maelezo ya kina), mifumo ya mabishano ya wakala wengi.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Mfano 1: Lango la kabla ya kitendo

Kipande cha HITL cha README huitisha wakala kwanza, kisha huomba mtumiaji kuidhinisha matokeo. Hilo ni mtiririko wa **baada ya kitendo**. Wakala tayari ameendesha, hivyo gharama ya wito wa LLM tayari imelipwa, na athari yoyote ya pembeni (barua pepe iliyotumwa, mstari wa hifadhidata uliyoandikwa, maoni yaliyowekwa) tayari imetokea.

Mtiririko wa **kabla ya kitendo** huingiza lango kabla wakala hajafanya hatua hatarishi. Wakala hupendekeza kitendo, lango hufanya uamuzi kama chaendeshe, na athari ya pembeni hutokea tu baada ya idhini.

| Kipengele | Uidhinishaji wa baada ya kitendo (kipande cha README) | Lango la kabla ya kitendo (daftari hili) |
|---|---|---|
| Idhini huendeshwaje? | Baada ya wakala kutoa matokeo | Kabla ya athari yoyote ya pembeni kutekelezwa |
| Gharama ya LLM kwa kukataliwa | Tayari imelipwa | Imelipwa kwa pendekezo tu, sio kitendo |
| Athari zisizorejelewa | Inawezekana (kitendo tayari kimetokea) | Zimezuia |
| Uwazi wa ukaguzi | Idhini ni tamko la chapa | Idhini ni rekodi ya JSON yenye alama ya wakati, kitendo, sababu |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Muundo 2: Ugawaji wa hatari

Si kila hatua inahitaji idhini ya binadamu. Kutafuta tu kwa kusoma tu dhidi ya API ya umma kuna hatari tofauti na kutuma barua pepe kwa mteja. Kutenda kama zote ni sawa kunapoteza umakini wa mtendaji na kunachelewesha wakala.

Mfano rahisi wa ngazi 3:

| Ngazi | Mifano | Mtiririko wa idhini |
|---|---|---|
| `chini` (kusoma tu) | Tafuta katika hifadhidata ya maarifa, angalia chaguzi za ndege, pata ukurasa wa wavuti wa umma | Imekamilishwa moja kwa moja, imerekodiwa kwa ukaguzi |
| `katikati` (mabadiliko ya gharama nafuu) | Hifadhi matokeo, ongeza kielekezi, panga kikumbusho | Imekamilishwa moja kwa moja, lakini hakiki ya kila siku ya kundi |
| `juu` (inayokabiliwa na nje au isiyoweza kubadilika) | Tuma barua pepe, kata kadi, chapisha kwenye kituo cha umma | Zuia hadi idhini ya binadamu |

Huu ni ugawaji mmoja wa ngazi. Mifumo ya uendeshaji mara nyingi hutumia ngazi nyembamba zaidi (k.m., ngazi za ruhusa za AWS IAM, ngazi za upatikanaji wa kulingana na nafasi). Toleo la ngazi 3 hapa chini ni toleo ndogo zaidi la maana kwa wakala anayechanganya hatua za kusoma tu na zinazosababisha athari.

Kainifaya hapa chini hutumia heuristics za maneno muhimu ili maonyesho yabaki kuwa ya uhakika na nafuu. Katika mfumo wa uendeshaji ungebadilisha hii kwa kainifaya aliyefunzwa au injini ya sera.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Muundo 3: Kumbukumbu ya ukaguzi na mzunguko wa marekebisho

`print("Response approved.")` si kumbukumbu ya ukaguzi. Kwa ajili ya kuaminika, kila uamuzi wa mlango unapaswa kurekodiwa kama tukio lililopangwa ambalo unaweza baadaye kuuliza, kurudia, au kuambatisha katika ukaguzi wa tukio.

Sehemu mbili:

1. **JSONL inayoongezwa tu.** Mstari mmoja kwa kila uamuzi, ikiwa na tarehe na saa, kitendo, daraja, uamuzi, sababu. Rahisi kuchuja, rahisi kuhifadhi katika hifadhi halisi ya kumbukumbu baadaye.
2. **Mzunguko wa marekebisho wakati wa kukataa.** Wakati mlango unarudisha `deny`, wakala hujiuliza tena kwa sababu ya kukataliwa ikilinganishwa, ili pendekezo linalofuata liweze kuepuka tatizo hilo.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Rasilimali za ziada

Miradi mingine kadhaa ya umma hutatua tofauti za mifumo hii ya HITL. Linganisha mbinu ili uone inayofaa kwa stack yako:

- **LangChain** vifungashio vya zana za mwanadamu-katika-mzunguko ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): vifungashio vya zana vinavyosimamisha utekelezaji kwa ajili ya ingizo la mwanadamu.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ imeibadilisha hii): hutumia nafasi ya wakala mahsusi kuwakilisha mwanadamu katika mazungumzo ya mawakala wengi.
- **Semantic Kernel** vichujio vya kazi ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware inayofanya kazi kila mwito wa kazi, inafaa kwa mantiki ya kukata.

Kila mradi hushughulikia mifumo midogo mitatu tofauti: LangChain huifunga kama zana, AutoGen hutumia nafasi ya wakala, Semantic Kernel hutumia vichujio vya middleware. Soma utekelezaji mmoja au miwili kutoka mwanzo hadi mwisho kabla ya kuchagua muundo kwa wakala wako mwenyewe.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
